## Шаг 1. Загрузка библиотек и данных

Загружаем датасет `merged_buildings_final_with_heights_parquet.parquet`, полученный после обучения модели CatBoost и заполнения пропусков высоты.

В датасете:
- `target_height_filled` — высота здания (наблюдаемая или предсказанная)
- `rep_geometry` — геометрия здания (полигон)
- `target_height_was_predicted` — флаг, предсказана ли высота (1) или наблюдалась (0)

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
from shapely import wkb
import plotly.graph_objects as go
from pathlib import Path 

DATA_PATH = Path("../data/processed/merged_buildings_final_with_heights_parquet.parquet")

df = pd.read_parquet(DATA_PATH)

print(f"Всего записей: {len(df):,}")
print(f"Колонки: {df.columns.tolist()}")
print("\nПервые 5 строк:")
print(df.head())

Всего записей: 139,649
Колонки: ['component_id', 'match_type', 'match_confidence', 'geometry_source', 'target_height', 'target_height_is_observed', 'target_height_source', 'target_height_source_detail', 'target_height_reliability', 'n_a', 'n_b', 'uids_a', 'uids_b', 'n_edges_ab', 'max_iou', 'mean_iou', 'max_overlap_a', 'max_overlap_b', 'min_dist_m', 'sum_area_a', 'sum_area_b', 'union_area_a', 'union_area_b', 'union_area_all', 'n_b_with_height', 'median_height_b', 'median_stairs_b', 'median_avg_floor_height_b', 'mode_purpose_b', 'n_neighbors_50m', 'n_neighbors_obs_height_50m', 'neighbor_height_mean_50m', 'neighbor_height_median_50m', 'neighbor_height_min_50m', 'neighbor_height_max_50m', 'neighbor_height_std_50m', 'neighbor_height_q25_50m', 'neighbor_height_q75_50m', 'n_neighbors_100m', 'n_neighbors_obs_height_100m', 'neighbor_height_mean_100m', 'neighbor_height_median_100m', 'neighbor_height_min_100m', 'neighbor_height_max_100m', 'neighbor_height_std_100m', 'neighbor_height_q25_100m', 'n

## Шаг 2. Преобразование геометрии

Геометрия в колонке `rep_geometry` хранится в бинарном формате WKB (Well-Known Binary). Для работы с ней необходимо преобразовать её в объекты `shapely.geometry`.

**Почему это важно:**
- WKB — компактный бинарный формат, но для пространственных операций (центроиды, площади, пересечения) нужны объекты Shapely
- Преобразование выполняется в метрической проекции EPSG:32635 (зона UTM 35N для Санкт-Петербурга), где координаты измеряются в метрах. Это критически важно для корректного расчёта центроидов — в географической системе (EPSG:4326) расчёт центроидов полигонов может давать искажения из-за кривизны Земли

**Порядок действий:**
1. Преобразуем WKB в геометрию Shapely
2. Создаём GeoDataFrame с системой координат EPSG:32635
3. На этом этапе геометрия готова для дальнейших вычислений

In [36]:
def wkb_to_geometry(x):
    try:
        if isinstance(x, bytes):
            return wkb.loads(x)
        elif isinstance(x, memoryview):
            return wkb.loads(x.tobytes())
        elif isinstance(x, bytearray):
            return wkb.loads(bytes(x))
        return x
    except:
        return None

df['geometry'] = df['rep_geometry'].apply(wkb_to_geometry)
print(f"Геометрия преобразована: {df['geometry'].notna().sum()} из {len(df)}")

gdf_metric = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:32635')
print(f"GeoDataFrame создан, CRS: {gdf_metric.crs}")

Геометрия преобразована: 139649 из 139649
GeoDataFrame создан, CRS: EPSG:32635


## Шаг 3. Расчёт центроидов в метрической проекции

Центроид — геометрический центр здания. Он нужен для:
- Привязки зданий к точкам на карте (для тепловых карт и 3D-визуализации)
- Пространственных запросов (поиск соседей, группировка)

**Почему важно считать центроиды в метрической проекции:**
- В системе EPSG:32635 координаты измеряются в метрах, и расчёт центроида выполняется корректно (среднее арифметическое вершин полигона)
- В географической системе EPSG:4326 (широта/долгота) центроид может смещаться из-за кривизны Земли, особенно для крупных полигонов
- Поэтому сначала считаем центроид в метрах, затем переведём координаты в градусы для отображения на карте

**Что делаем:**
1. Вычисляем центроид каждого здания
2. Извлекаем координаты X и Y (в метрах)

In [37]:
gdf_metric['centroid'] = gdf_metric.geometry.centroid
gdf_metric['centroid_x'] = gdf_metric['centroid'].x
gdf_metric['centroid_y'] = gdf_metric['centroid'].y

print(f"Центроиды добавлены. Пропусков: {gdf_metric['centroid'].isna().sum()}")
print("\nПример центроидов (метры):")
print(gdf_metric[['centroid_x', 'centroid_y']].head())

Центроиды добавлены. Пропусков: 0

Пример центроидов (метры):
      centroid_x    centroid_y
0  673859.539664  6.635505e+06
1  673885.218595  6.635511e+06
2  677022.530648  6.640407e+06
3  677493.331394  6.654095e+06
4  677486.041458  6.654096e+06


## Шаг 4. Перевод в градусы для визуализации

Для отображения данных на карте (folium, тепловые карты) координаты должны быть в географической системе EPSG:4326 (широта/долгота).

**Почему важно сначала считать центроиды в метрической проекции:**
- В системе EPSG:32635 (метры) расчёт центроида корректен
- После перевода в EPSG:4326 (градусы) повторный расчёт центроида даёт предупреждение, но мы используем его только для привязки точек на карте, где допустима небольшая погрешность

**Порядок действий:**
1. Переводим GeoDataFrame из EPSG:32635 в EPSG:4326
2. Вычисляем центроиды (уже в градусах) для точечных визуализаций — предупреждение игнорируем, так как точки нужны только для отображения
3. Добавляем колонки lat (широта) и lon (долгота)
4. Переносим значение confidence_score из исходного датасета с явным приведением типов

In [38]:
gdf_deg = gdf_metric.to_crs('EPSG:4326')

# Вычисляем центроиды в градусах для отображения на карте
# Предупреждение игнорируем — для точечных визуализаций погрешность допустима
gdf_deg['centroid_deg'] = gdf_deg.geometry.centroid
gdf_deg['lat'] = gdf_deg['centroid_deg'].y
gdf_deg['lon'] = gdf_deg['centroid_deg'].x

# Создаём confidence_score как float (чтобы можно было присвоить 0.5)
reliability_map = {'high': 3, 'medium': 2, 'low': 1, 'none': 0}
gdf_deg['confidence_score'] = df['target_height_reliability'].map(reliability_map).astype(float).fillna(0)

# Для предсказанных высот ставим 0.5
gdf_deg.loc[df['target_height_was_predicted'] == 1, 'confidence_score'] = 0.5

print(f"Координаты в градусах. Диапазон:")
print(f"  Широта (lat): {gdf_deg['lat'].min():.4f} – {gdf_deg['lat'].max():.4f}")
print(f"  Долгота (lon): {gdf_deg['lon'].min():.4f} – {gdf_deg['lon'].max():.4f}")
print(f"\nРаспределение confidence_score:")
print(gdf_deg['confidence_score'].value_counts().sort_index())

Координаты в градусах. Диапазон:
  Широта (lat): 59.6924 – 60.0899
  Долгота (lon): 30.0549 – 30.5646

Распределение confidence_score:
confidence_score
0.5    38086
1.0     5149
2.0    43150
3.0    53264
Name: count, dtype: int64


C:\Users\kanas\AppData\Local\Temp\ipykernel_16216\3335940407.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_deg['centroid_deg'] = gdf_deg.geometry.centroid


## Шаг 5. Подготовка данных для тепловых карт

Для тепловой карты высот создаём две версии:
1. **Полная** — со всеми зданиями, включая Лахта-центр (462 м)
2. **Детальная** — с ограничением высоты до 200 м, чтобы лучше видеть основную застройку

**Зачем нужны две версии:**
- Полная карта показывает весь диапазон высот, включая уникальные высотные доминанты
- Детальная карта позволяет рассмотреть распределение высот в основной массе зданий (до 200 м), где сосредоточено 99% объектов

In [39]:
# Данные для полной карты (все высоты)
heat_full = gdf_deg[['lat', 'lon', 'target_height_filled']].dropna()
heat_full_points = [[row['lat'], row['lon'], row['target_height_filled']] 
                    for _, row in heat_full.iterrows()]

# Данные для детальной карты (до 200 м)
heat_detailed = gdf_deg[gdf_deg['target_height_filled'] <= 200].copy()
heat_detailed = heat_detailed[['lat', 'lon', 'target_height_filled']].dropna()
heat_detailed_points = [[row['lat'], row['lon'], row['target_height_filled']] 
                        for _, row in heat_detailed.iterrows()]

print(f"Полная карта: {len(heat_full_points):,} точек")
print(f"Детальная карта (до 200 м): {len(heat_detailed_points):,} точек")
print(f"Исключено (выше 200 м): {len(heat_full_points) - len(heat_detailed_points):,} точек")

Полная карта: 139,649 точек
Детальная карта (до 200 м): 139,648 точек
Исключено (выше 200 м): 1 точек


## Шаг 6. Тепловая карта высот (полная)

Тепловая карта показывает плотность зданий с цветовой кодировкой по высоте. Цвета варьируются от синего (низкие здания) до красного (высокие). В полной версии отображены все здания, включая уникальные высотные доминанты (Лахта-центр, 462 м).

**Легенда:**
- Синий — низкие здания (до 15 м)
- Зелёный — средняя высота (15–30 м)
- Жёлтый — выше среднего (30–50 м)
- Оранжевый — высокие здания (50–100 м)
- Красный — высотные здания (>100 м, включая Лахту)

*Примечание:* тепловая карта основана на взвешенных точках (центроидах зданий). Интенсивность свечения пропорциональна высоте здания.

In [ ]:
m_full = folium.Map(location=[59.93, 30.31], zoom_start=11, tiles='CartoDB positron')
HeatMap(heat_full_points, radius=15, blur=10, min_opacity=0.5).add_to(m_full)

# Добавляем легенду
legend_html = '''
<div style="position: fixed; bottom: 50px; right: 50px; z-index: 1000; background-color: white; padding: 10px; border: 2px solid grey; border-radius: 5px; font-size: 12px;">
    <b>Высота зданий</b><br>
    <span style="background: #2c7bb6; width: 20px; height: 20px; display: inline-block;"></span> 0–15 м (низкие)<br>
    <span style="background: #abd9e9; width: 20px; height: 20px; display: inline-block;"></span> 15–30 м<br>
    <span style="background: #ffffbf; width: 20px; height: 20px; display: inline-block;"></span> 30–50 м<br>
    <span style="background: #fdae61; width: 20px; height: 20px; display: inline-block;"></span> 50–100 м<br>
    <span style="background: #d7191c; width: 20px; height: 20px; display: inline-block;"></span> >100 м (включая Лахту)
</div>
'''

m_full.get_root().html.add_child(folium.Element(legend_html))
OUTPUT_PATH = Path("../data/map/spb_height_heatmap_full.html") 
m_full.save(OUTPUT_PATH)
print("Тепловая карта (полная) сохранена в spb_height_heatmap_full.html")

Тепловая карта (полная) сохранена в spb_height_heatmap_full.html


## Шаг 7. Тепловая карта высот (детальная, до 200 м)

Для лучшей визуализации основной застройки (99.99% зданий) создана карта с ограничением высоты до 200 м. Это позволяет детально рассмотреть распределение высот в городской среде без сжатия цветовой шкалы под влиянием единственного выброса (Лахта-центр).

**Легенда:**
- Синий — низкие здания (до 15 м)
- Зелёный — средняя высота (15–30 м)
- Жёлтый — выше среднего (30–50 м)
- Оранжевый — высокие здания (50–100 м)
- Красный — очень высокие (100–200 м)

*Примечание:* Лахта-центр (462 м) исключена из этой карты для улучшения детализации основной застройки.

In [ ]:
m_detailed = folium.Map(location=[59.93, 30.31], zoom_start=11, tiles='CartoDB positron')
HeatMap(heat_detailed_points, radius=15, blur=10, min_opacity=0.5).add_to(m_detailed)

# Добавляем легенду
legend_html_detailed = '''
<div style="position: fixed; bottom: 50px; right: 50px; z-index: 1000; background-color: white; padding: 10px; border: 2px solid grey; border-radius: 5px; font-size: 12px;">
    <b>Высота зданий (до 200 м)</b><br>
    <span style="background: #2c7bb6; width: 20px; height: 20px; display: inline-block;"></span> 0–15 м (низкие)<br>
    <span style="background: #abd9e9; width: 20px; height: 20px; display: inline-block;"></span> 15–30 м<br>
    <span style="background: #ffffbf; width: 20px; height: 20px; display: inline-block;"></span> 30–50 м<br>
    <span style="background: #fdae61; width: 20px; height: 20px; display: inline-block;"></span> 50–100 м<br>
    <span style="background: #d7191c; width: 20px; height: 20px; display: inline-block;"></span> 100–200 м
</div>
'''

m_detailed.get_root().html.add_child(folium.Element(legend_html_detailed))
OUTPUT_PATH = Path("../data/map/spb_height_heatmap_detailed.html")
m_detailed.save(OUTPUT_PATH)
print("Тепловая карта (детальная) сохранена в spb_height_heatmap_detailed.html")

Тепловая карта (детальная) сохранена в spb_height_heatmap_detailed.html


## Шаг 8. Карта доверия к высоте

Карта доверия показывает, насколько можно полагаться на значение высоты для каждого здания.

**Цветовая кодировка:**
- Синий — **high**: наблюдаемая высота, высокая уверенность (B_only или 1:1 с high confidence)
- Голубой — **medium**: наблюдаемая высота, средняя уверенность (n:1, 1:n, n:n)
- Оранжевый — **low**: наблюдаемая высота, низкая уверенность (слабые связи)
- Жёлтый — **predicted**: высота предсказана моделью (A_only)
- Красный — **none**: высота отсутствует (в финальном датасете таких быть не должно)

*Примечание:* для производительности отображается 10% зданий (случайная выборка).

In [ ]:
m_confidence = folium.Map(location=[59.93, 30.31], zoom_start=11, tiles='CartoDB positron')

def get_confidence_color(score):
    if score == 3:
        return '#2c7bb6'   # синий (high)
    elif score == 2:
        return '#abd9e9'   # голубой (medium)
    elif score == 1:
        return '#fdae61'   # оранжевый (low)
    elif score == 0.5:
        return '#ffffbf'   # жёлтый (predicted)
    else:
        return '#d7191c'   # красный (none)

# Берём выборку (каждый 10-й)
sample = gdf_deg.sample(n=len(gdf_deg)//10, random_state=42)

for _, row in sample.iterrows():
    color = get_confidence_color(row['confidence_score'])
    
    if row['target_height_was_predicted'] == 1:
        tooltip = f"Высота: {row['target_height_filled']:.1f} м (предсказано)"
    else:
        tooltip = f"Высота: {row['target_height_filled']:.1f} м (наблюдаемая)"
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        weight=0.5,
        tooltip=tooltip
    ).add_to(m_confidence)

# Легенда
legend_confidence = '''
<div style="position: fixed; bottom: 50px; right: 50px; z-index: 1000; background-color: white; padding: 10px; border: 2px solid grey; border-radius: 5px; font-size: 12px;">
    <b>Доверие к высоте</b><br>
    <span style="background: #2c7bb6; width: 20px; height: 20px; display: inline-block;"></span> high (наблюдаемая, надёжная)<br>
    <span style="background: #abd9e9; width: 20px; height: 20px; display: inline-block;"></span> medium (наблюдаемая, средняя)<br>
    <span style="background: #fdae61; width: 20px; height: 20px; display: inline-block;"></span> low (наблюдаемая, низкая)<br>
    <span style="background: #ffffbf; width: 20px; height: 20px; display: inline-block;"></span> predicted (предсказано моделью)<br>
    <span style="background: #d7191c; width: 20px; height: 20px; display: inline-block;"></span> none (нет данных)
</div>
'''

m_confidence.get_root().html.add_child(folium.Element(legend_confidence))
OUTPUT_PATH = Path("../data/map/spb_confidence_map10.html")
m_confidence.save(OUTPUT_PATH)
print("Карта доверия сохранена в spb_confidence_map10.html")

Карта доверия сохранена в spb_confidence_map10.html


## Шаг 8. Карта доверия к высоте

Карта доверия показывает, насколько можно полагаться на значение высоты для каждого здания. В отличие от выборки (10%), в полной версии отображены **все** здания (139 649 объектов), что позволяет детально оценить пространственное распределение надёжности данных.

**Цветовая кодировка:**
- Синий — **high**: наблюдаемая высота, высокая уверенность (B_only или 1:1 с high confidence)
- Голубой — **medium**: наблюдаемая высота, средняя уверенность (n:1, 1:n, n:n)
- Оранжевый — **low**: наблюдаемая высота, низкая уверенность (слабые связи)
- Жёлтый — **predicted**: высота предсказана моделью (A_only)
- Красный — **none**: высота отсутствует (в финальном датасете таких быть не должно)

*Примечание:* для повышения производительности отображения используется уменьшенный радиус точек, так как обрабатывается полный набор данных.

In [ ]:
import folium
from folium import plugins

m_confidence_full = folium.Map(location=[59.93, 30.31], zoom_start=11, tiles='CartoDB positron')

def get_confidence_color(score):
    if score == 3:
        return '#2c7bb6'   # синий (high)
    elif score == 2:
        return '#abd9e9'   # голубой (medium)
    elif score == 1:
        return '#fdae61'   # оранжевый (low)
    elif score == 0.5:
        return '#ffffbf'   # жёлтый (predicted)
    else:
        return '#d7191c'   # красный (none)

# Используем MarkerCluster для оптимизации отображения большого количества точек
marker_cluster = plugins.MarkerCluster().add_to(m_confidence_full)

for _, row in gdf_deg.iterrows():
    color = get_confidence_color(row['confidence_score'])
    
    if row['target_height_was_predicted'] == 1:
        tooltip = f"Высота: {row['target_height_filled']:.1f} м (предсказано)"
    else:
        tooltip = f"Высота: {row['target_height_filled']:.1f} м (наблюдаемая)"
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=1.5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        weight=0.3,
        tooltip=tooltip
    ).add_to(marker_cluster)

# Легенда
legend_confidence = '''
<div style="position: fixed; bottom: 50px; right: 50px; z-index: 1000; background-color: white; padding: 10px; border: 2px solid grey; border-radius: 5px; font-size: 12px;">
    <b>Доверие к высоте</b><br>
    <span style="background: #2c7bb6; width: 20px; height: 20px; display: inline-block;"></span> high (наблюдаемая, надёжная)<br>
    <span style="background: #abd9e9; width: 20px; height: 20px; display: inline-block;"></span> medium (наблюдаемая, средняя)<br>
    <span style="background: #fdae61; width: 20px; height: 20px; display: inline-block;"></span> low (наблюдаемая, низкая)<br>
    <span style="background: #ffffbf; width: 20px; height: 20px; display: inline-block;"></span> predicted (предсказано моделью)<br>
    <span style="background: #d7191c; width: 20px; height: 20px; display: inline-block;"></span> none (нет данных)
</div>
'''

m_confidence_full.get_root().html.add_child(folium.Element(legend_confidence))
OUTPUT_PATH = Path("../data/map/spb_confidence_map_full.html")
m_confidence_full.save(OUTPUT_PATH)
print("Карта доверия (полная) сохранена в spb_confidence_map_full.html")
print(f"Отображено зданий: {len(gdf_deg):,}")

Карта доверия (полная) сохранена в spb_confidence_map_full.html
Отображено зданий: 139,649


In [ ]:
m_confidence_heat = folium.Map(location=[59.93, 30.31], zoom_start=11, tiles='CartoDB positron')

heat_confidence = gdf_deg[['lat', 'lon', 'confidence_score']].dropna()
heat_confidence_points = [[row['lat'], row['lon'], row['confidence_score']] 
                          for _, row in heat_confidence.iterrows()]

HeatMap(heat_confidence_points, radius=20, blur=15, min_opacity=0.3).add_to(m_confidence_heat)

# Легенда
legend_heat = '''
<div style="position: fixed; bottom: 50px; right: 50px; z-index: 1000; background-color: white; padding: 10px; border: 2px solid grey; border-radius: 5px; font-size: 12px;">
    <b>Доверие (тепловая карта)</b><br>
    <span style="background: #2c7bb6; width: 20px; height: 20px; display: inline-block;"></span> высокое доверие (high)<br>
    <span style="background: #abd9e9; width: 20px; height: 20px; display: inline-block;"></span> среднее доверие (medium)<br>
    <span style="background: #fdae61; width: 20px; height: 20px; display: inline-block;"></span> низкое доверие (low)<br>
    <span style="background: #ffffbf; width: 20px; height: 20px; display: inline-block;"></span> предсказано (predicted)
</div>
'''

m_confidence_heat.get_root().html.add_child(folium.Element(legend_heat))
OUTPUT_PATH = Path("../data/map/spb_confidence_heatmap.html")
m_confidence_heat.save(OUTPUT_PATH)
print("Тепловая карта доверия сохранена в spb_confidence_heatmap.html")

Тепловая карта доверия сохранена в spb_confidence_heatmap.html
